<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/10-miniproyecto-reporte-de-la-manana/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.10 · Miniproyecto: el reporte de la mañana

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas celdas. Dejan listos `arbol`, `conteo`, `lector`, `pronostico` y la función `reporte_de_la_manana`.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
FERIADOS = pd.to_datetime(
    [
        "2024-01-01",
        "2024-03-28",
        "2024-03-29",
        "2024-05-01",
        "2024-06-29",
        "2024-07-28",
        "2024-07-29",
        "2024-08-30",
        "2024-10-08",
        "2024-11-01",
        "2024-12-08",
        "2024-12-09",
        "2024-12-25",
        "2025-01-01",
        "2025-04-17",
        "2025-04-18",
        "2025-05-01",
        "2025-06-29",
        "2025-07-28",
        "2025-07-29",
        "2025-08-30",
        "2025-10-08",
        "2025-11-01",
        "2025-12-08",
        "2025-12-09",
        "2025-12-25",
    ]
)

In [3]:
def calendario(dias):
    """Una columna por día de la semana y por mes, más la tendencia: los días desde el inicio."""
    x = pd.DataFrame(index=dias)
    for dia in range(7):
        x[f"dia_{dia}"] = (dias.dayofweek == dia).astype(int)
    for mes in range(1, 13):
        x[f"mes_{mes}"] = (dias.month == mes).astype(int)
    x["tendencia"] = (dias - pd.Timestamp("2024-01-01")).days
    return x

In [4]:
def tres_pronosticos(serie, feriados, mes):
    """Pronostica cada día de un mes con tres métodos, usando solo lo anterior a ese mes."""
    inicio = pd.Timestamp(mes + "-01")
    dias = pd.date_range(inicio, inicio + pd.offsets.MonthEnd(0))
    antes = serie[serie.index < inicio]
    laborables = antes[~antes.index.isin(feriados)]

    # Promedio por día de la semana de las últimas 8 semanas.
    ultimas = laborables[laborables.index >= inicio - pd.Timedelta(days=56)]
    por_dia = ultimas.groupby(ultimas.index.dayofweek).mean()
    ocho_semanas = pd.Series(dias.dayofweek.map(por_dia), index=dias)

    # Índice: cuánto subió ese mes en 2024 frente a las 8 semanas previas de 2024.
    hace_un_anio = inicio - pd.Timedelta(days=364)
    base = laborables[
        (laborables.index >= hace_un_anio - pd.Timedelta(days=56))
        & (laborables.index < hace_un_anio)
    ]
    mismo_mes = laborables[
        (laborables.index.year == 2024) & (laborables.index.month == inicio.month)
    ]
    regla = ocho_semanas * mismo_mes.mean() / base.mean()

    modelo = LinearRegression().fit(calendario(laborables.index), laborables)
    regresion = pd.Series(modelo.predict(calendario(dias)), index=dias)

    # Todos los métodos ponen los feriados en cero: esos días no se vende.
    tabla = pd.DataFrame(
        {"real": serie[dias], "ocho_semanas": ocho_semanas, "regla": regla, "regresion": regresion}
    )
    metodos = ["ocho_semanas", "regla", "regresion"]
    tabla.loc[tabla.index.isin(feriados), metodos] = 0
    return tabla

In [5]:
def preparar_despachos(despachos):
    """Deja los datos que se conocen al preparar un despacho: textos como 0 y 1, mes y día."""
    fecha = pd.to_datetime(despachos["fecha"])
    x = pd.get_dummies(despachos[["ciudad", "tipo", "transportista"]], dtype=int)
    x = x.drop(columns=["ciudad_Lima", "tipo_Bodega", "transportista_Propio"], errors="ignore")
    x["mes"] = fecha.dt.month
    x["dia"] = fecha.dt.dayofweek
    return x.join(despachos[["distancia_km", "lineas", "presentaciones"]])

In [6]:
ventas = pd.read_csv(DATOS + "ventas.csv", parse_dates=["fecha"])
despachos = pd.read_csv(DATOS + "despachos.csv")
tickets = pd.read_csv(DATOS + "tickets.csv")

serie = ventas.groupby("fecha")["total"].sum().asfreq("D", fill_value=0)

In [7]:
previos = despachos[despachos["fecha"] < "2025-12-01"]
arbol = DecisionTreeClassifier(max_depth=6, random_state=0)
arbol.fit(preparar_despachos(previos), previos["llego_tarde"])

mensajes = tickets[tickets["fecha"] < "2025-12-01"]
conteo = CountVectorizer()
lector = LogisticRegression(max_iter=2000)
lector.fit(conteo.fit_transform(mensajes["mensaje"]), mensajes["categoria"])

pronostico = tres_pronosticos(serie, FERIADOS, "2025-12")

In [8]:
def reporte_de_la_manana(dia):
    """Lo que se sabe esa mañana: venta esperada, despachos a vigilar y tickets de ayer por área."""
    ayer = str((pd.Timestamp(dia) - pd.Timedelta(days=1)).date())
    hoy = despachos[despachos["fecha"] == dia].sample(frac=1, random_state=0)
    columnas = ["id_despacho", "id_cliente", "ciudad", "transportista", "riesgo"]
    if hoy.empty:  # Feriado: no hay despachos.
        vigilar = pd.DataFrame(columns=columnas)
    else:
        x = preparar_despachos(hoy).reindex(columns=arbol.feature_names_in_, fill_value=0)
        riesgo = arbol.predict_proba(x)[:, 1].round(2)
        cupo = max(1, round(len(hoy) * 0.10))
        vigilar = hoy.assign(riesgo=riesgo).nlargest(cupo, "riesgo")[columnas]

    de_ayer = tickets[tickets["fecha"] == ayer]
    por_area = de_ayer.assign(area="")[["id_ticket", "area", "mensaje"]]
    if not de_ayer.empty:
        por_area["area"] = lector.predict(conteo.transform(de_ayer["mensaje"]))

    venta = pronostico.loc[dia, ["regla", "regresion"]].round(0)
    return venta, vigilar, por_area

### 1. Otra mañana

Arma el reporte del `"2025-12-22"`. ¿Cuántos despachos hay que vigilar? Deben ser `8`.

In [9]:
venta, vigilar, por_area = reporte_de_la_manana("2025-12-22")

len(vigilar)

8

### 2. ¿Y un feriado?

Mira `pronostico.loc["2025-12-25"]` y `reporte_de_la_manana("2025-12-25")`. ¿Qué dice cada método? ¿Qué trae el reporte? ¿Por qué?

In [10]:
print(pronostico.loc["2025-12-25"])
print(reporte_de_la_manana("2025-12-25")[1])

# Todos dicen cero, y no hay despachos que vigilar, porque es feriado.

real            0.0
ocho_semanas    0.0
regla           0.0
regresion       0.0
Name: 2025-12-25 00:00:00, dtype: float64
Empty DataFrame
Columns: [id_despacho, id_cliente, ciudad, transportista, riesgo]
Index: []


### 3. Comprobar esa mañana

Del reporte del `"2025-12-22"`, ¿cuántos de los despachos vigilados llegaron tarde? Debe dar `6`.

> 💡 `despachos["id_despacho"].isin(vigilar["id_despacho"])` marca sus filas, y luego sumas `llego_tarde`.

In [11]:
print(despachos[despachos["id_despacho"].isin(vigilar["id_despacho"])]["llego_tarde"].sum())

6


### 4. Un modelo sin reentrenar

Entrena un lector solo con los mensajes anteriores al `"2025-07-01"`, con `CountVectorizer` y `LogisticRegression(max_iter=2000)`. Calcula su exactitud en los mensajes de diciembre. Debe dar `0.845`.

In [12]:
antiguos = tickets[tickets["fecha"] < "2025-07-01"]
de_diciembre = tickets[tickets["fecha"] >= "2025-12-01"]

conteo_antiguo = CountVectorizer()
sin_reentrenar = LogisticRegression(max_iter=2000)
sin_reentrenar.fit(conteo_antiguo.fit_transform(antiguos["mensaje"]), antiguos["categoria"])

x_diciembre = conteo_antiguo.transform(de_diciembre["mensaje"])
round(sin_reentrenar.score(x_diciembre, de_diciembre["categoria"]), 3)

0.845

### 5. Tu Excel

Guarda el reporte del `"2025-12-22"` en un archivo `reporte_2025_12_22.xlsx`, con hojas `Despachos` y `Tickets`.

In [13]:
with pd.ExcelWriter("reporte_2025_12_22.xlsx", engine="openpyxl") as archivo:
    vigilar.to_excel(archivo, sheet_name="Despachos", index=False)
    por_area.to_excel(archivo, sheet_name="Tickets", index=False)

### 6. ¿Qué le dirías a tu jefe?

Responde en un comentario. Tu jefe quiere despedir a quien revisa los despachos, porque "ahora lo hace el modelo". Con lo que viste en la clase, ¿qué le contestas?

In [14]:
# Que el modelo solo ordena qué mirar primero. Se le escapan 6 de cada 10 tardíos, y alguien
# tiene que llamar al transportista y avisar al cliente. Además, es un solo mes y datos sintéticos.

## 🎉 ¡Terminaste la Temporada 2!

Nos vemos en la **Temporada 3**, con un proyecto completo.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0